# CUDA Static Cluster-Pool FFN Training Notebook

This notebook is the CUDA-side training gate for the locked FFN-v1 primitive:

```text
static cluster-pool sparse FFN
C = 16 clusters
M = 192 candidate neurons per cluster
dense attention kept on
no MacroV2
```

It follows the same Colab/VSCode pattern as the kernel benchmark notebook: the remote runtime finds or clones the repo, installs it editable, checks that the required CLI commands exist, then runs the training/eval commands from the repo code.

No old checkpoints are assumed to exist in GitHub. By default this notebook trains a dense checkpoint on the remote runtime first. To use an existing remote checkpoint, set `DENSE_CHECKPOINT=/path/on/colab/checkpoint.pt` before running and set `RUN_DENSE_TRAIN=False` in the settings cell.

In [ ]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any


def _run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(str(x) for x in cmd), flush=True)
    subprocess.check_call([str(x) for x in cmd], cwd=str(cwd) if cwd else None)


def _capture(cmd: list[str], *, cwd: Path | None = None) -> str:
    print("$", " ".join(str(x) for x in cmd), flush=True)
    return subprocess.check_output(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        stderr=subprocess.STDOUT,
        text=True,
    )


def _checkout_repo_ref(repo: Path, repo_ref: str) -> Path:
    if not (repo / ".git").exists():
        return repo.resolve()
    if os.environ.get("SPEEDUP_THINGY_UPDATE", "1").strip().lower() in {"0", "false", "no", "off"}:
        return repo.resolve()
    _run(["git", "fetch", "origin"], cwd=repo)
    _run(["git", "checkout", repo_ref], cwd=repo)
    try:
        _run(["git", "pull", "--ff-only"], cwd=repo)
    except subprocess.CalledProcessError:
        print("git pull skipped; likely checked out a detached commit/ref", flush=True)
    return repo.resolve()


def find_or_clone_repo() -> Path:
    repo_ref = os.environ.get("SPEEDUP_THINGY_REF", "master")

    # If the user explicitly points at a checkout, trust that checkout. This is useful for
    # mounted workspaces or a manually prepared Colab folder.
    env_root = os.environ.get("SPEEDUP_THINGY_REPO")
    if env_root:
        candidate = Path(env_root).expanduser()
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    # If the notebook is already running from a repo checkout, use it without mutating it.
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent):
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    clone_dir = Path(os.environ.get("SPEEDUP_THINGY_CLONE_DIR", "/content/Speedup-Thingy"))
    repo_url = os.environ.get("SPEEDUP_THINGY_REPO_URL", "https://github.com/BrownHujay/Speedup-Thingy.git")
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    if not clone_dir.exists():
        try:
            _run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(clone_dir)])
        except subprocess.CalledProcessError:
            _run(["git", "clone", repo_url, str(clone_dir)])
            _run(["git", "checkout", repo_ref], cwd=clone_dir)
    elif (clone_dir / "src" / "recursive_training_engine").exists():
        return _checkout_repo_ref(clone_dir, repo_ref)
    else:
        raise RuntimeError(f"Clone directory exists but is not a Speedup-Thingy repo: {clone_dir}")
    return clone_dir.resolve()


repo_root = find_or_clone_repo()
src_path = repo_root / "src"
os.environ["PYTHONPATH"] = str(src_path) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    import yaml  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

try:
    import recursive_training_engine  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"])

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA Colab runtime before running this notebook.")

torch.backends.cuda.matmul.allow_tf32 = True


def rte_cmd(*args: object) -> list[str]:
    return [sys.executable, "-c", "from recursive_training_engine.cli import main; main()", *[str(a) for a in args]]


def run_rte(*args: object) -> None:
    _run(rte_cmd(*args), cwd=repo_root)


help_text = _capture(rte_cmd("--help"), cwd=repo_root)
required_commands = [
    "train",
    "static-cluster-pool-continuation",
    "static-cluster-pool-gradient-alignment",
    "benchmark-static-cluster-pool-ffn-train",
]
missing = [name for name in required_commands if name not in help_text]
if missing:
    raise RuntimeError(
        "This remote checkout does not contain the static cluster-pool training commands: "
        + ", ".join(missing)
        + "\nSet SPEEDUP_THINGY_REF to the branch/commit that has the latest kernels/CLI."
    )

print("repo_root:", repo_root)
print("torch:", torch.__version__)
print("cuda device:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))

## Settings

Defaults are deliberately CUDA-Colab friendly: `d_model=256`, `d_ff=2048`, synthetic tokens, 400 dense pretrain steps, then the 50/300-step continuation gates. Raise `D_MODEL` to `512` after the first pass if you want the larger-H validation from the guide.

The notebook writes every artifact under a timestamped remote `runs/colab_static_cluster_ffn_*` directory.

In [ ]:
def _bool_env(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def _int_list_env(name: str, default: list[int]) -> list[int]:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return list(default)
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


# Larger-H quality/training validation.
D_MODEL = int(os.environ.get("D_MODEL", "256"))
D_FF = int(os.environ.get("D_FF", "2048"))
N_LAYERS = int(os.environ.get("N_LAYERS", "6"))
N_HEADS = int(os.environ.get("N_HEADS", "8"))
VOCAB_SIZE = int(os.environ.get("VOCAB_SIZE", "2048"))
SEQ_LEN = int(os.environ.get("SEQ_LEN", "64"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "32"))
PRECISION = os.environ.get("PRECISION", "fp32")
SEED = int(os.environ.get("SEED", "1234"))

# FFN-v1 primitive locked by the previous gates.
CLUSTERS = int(os.environ.get("CLUSTERS", "16"))
CANDIDATE_M = int(os.environ.get("CANDIDATE_M", "192"))
SVD_RANK = int(os.environ.get("SVD_RANK", "64"))
CALIBRATION_TOKENS = int(os.environ.get("CALIBRATION_TOKENS", "8192"))
CLUSTER_ITERS = int(os.environ.get("CLUSTER_ITERS", "4"))

# Dense pretrain and continuation gates.
DENSE_CKPT_STEPS = _int_list_env("DENSE_CKPT_STEPS", [100, 200, 400])
CONT_LR = float(os.environ.get("CONT_LR", "1e-4"))
CONT_WEIGHT_DECAY = float(os.environ.get("CONT_WEIGHT_DECAY", "0.0"))
EVAL_TOKENS = int(os.environ.get("EVAL_TOKENS", "65536"))
EVAL_BATCHES = max(1, math.ceil(EVAL_TOKENS / (BATCH_SIZE * SEQ_LEN)))
ALIGNMENT_BATCHES = int(os.environ.get("ALIGNMENT_BATCHES", "1"))
ALIGNMENT_BATCH_SIZE = int(os.environ.get("ALIGNMENT_BATCH_SIZE", str(BATCH_SIZE)))

RUN_DENSE_TRAIN = _bool_env("RUN_DENSE_TRAIN", True)
RUN_50_CONTINUATION = _bool_env("RUN_50_CONTINUATION", True)
RUN_300_CONTINUATION = _bool_env("RUN_300_CONTINUATION", True)
RUN_FWD_BWD_BENCH = _bool_env("RUN_FWD_BWD_BENCH", True)
RUN_EARLY_HANDOFF = _bool_env("RUN_EARLY_HANDOFF", True)
RESUME_EXISTING = _bool_env("RESUME_EXISTING", True)

# Optional data switch. Synthetic is the default so this runs without dataset downloads.
USE_TINYSTORIES = _bool_env("USE_TINYSTORIES", False)
SYNTHETIC_TOKENS = int(os.environ.get("SYNTHETIC_TOKENS", "2000000"))

# If set, this remote checkpoint is used instead of training a dense checkpoint here.
DENSE_CHECKPOINT_OVERRIDE = os.environ.get("DENSE_CHECKPOINT", "").strip()

RUN_STAMP = os.environ.get("RUN_STAMP", time.strftime("%Y%m%d_%H%M%S"))
RUN_ROOT = repo_root / "runs" / f"colab_static_cluster_ffn_d{D_MODEL}_h{D_FF}_{RUN_STAMP}"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
REPORT_DIR = RUN_ROOT / "reports"
CONFIG_DIR = RUN_ROOT / "configs"
for path in (CHECKPOINT_DIR, REPORT_DIR, CONFIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

BENCH_SIZES = os.environ.get("BENCH_SIZES", "512x2048x1024,2048x8192x1024")
BENCH_SIZES = [x.strip() for x in BENCH_SIZES.split(",") if x.strip()]
BENCH_ITERS = int(os.environ.get("BENCH_ITERS", "5"))
BENCH_WARMUP = int(os.environ.get("BENCH_WARMUP", "1"))
EARLY_HANDOFF_STEPS = _int_list_env("EARLY_HANDOFF_STEPS", [100, 200])
EARLY_HANDOFF_CONT_STEPS = int(os.environ.get("EARLY_HANDOFF_CONT_STEPS", "300"))

settings = {
    "run_root": str(RUN_ROOT),
    "model": {"d_model": D_MODEL, "d_ff": D_FF, "layers": N_LAYERS, "heads": N_HEADS, "vocab": VOCAB_SIZE},
    "training": {"batch_size": BATCH_SIZE, "seq_len": SEQ_LEN, "precision": PRECISION, "seed": SEED},
    "ffn_v1": {"clusters": CLUSTERS, "candidate_m": CANDIDATE_M, "rank": SVD_RANK, "calibration_tokens": CALIBRATION_TOKENS},
    "eval_batches": EVAL_BATCHES,
    "dense_ckpt_steps": DENSE_CKPT_STEPS,
    "toggles": {
        "run_dense_train": RUN_DENSE_TRAIN,
        "run_50_continuation": RUN_50_CONTINUATION,
        "run_300_continuation": RUN_300_CONTINUATION,
        "run_fwd_bwd_bench": RUN_FWD_BWD_BENCH,
        "run_early_handoff": RUN_EARLY_HANDOFF,
    },
}
print(json.dumps(settings, indent=2))

if USE_TINYSTORIES:
    _run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "datasets"])

## Generate The Config

This writes a remote config under the run directory. The file is JSON with a `.yaml` suffix; JSON is valid YAML, and this avoids notebook-side YAML dependencies.

In [ ]:
def _divisor_at_most(value: int, limit: int) -> int:
    candidate = min(value, limit)
    while candidate > 1 and value % candidate != 0:
        candidate -= 1
    return max(1, candidate)


ffn_groups = _divisor_at_most(D_FF, 64)
head_groups = _divisor_at_most(N_HEADS, N_HEADS)
config = {
    "run_name": f"colab_static_cluster_ffn_d{D_MODEL}_h{D_FF}",
    "output_dir": str(RUN_ROOT),
    "model": {
        "topology": "dense",
        "vocab_size": VOCAB_SIZE,
        "d_model": D_MODEL,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "n_dense_layers": N_LAYERS,
        "n_prelude": 1,
        "n_coda": 1,
        "t_max": 4,
        "depth_choices": [1, 2, 4],
        "attn_banks": 1,
        "ffn_banks": 2,
        "head_groups": head_groups,
        "ffn_groups": ffn_groups,
        "active_head_groups": min(2, head_groups),
        "active_ffn_groups": min(4, ffn_groups),
        "recipe_count": 16,
        "tie_embeddings": True,
        "use_rope": True,
        "use_recursive_input_skip": True,
        "fairness_tolerance": 0.01,
        "macro_rank": max(1, min(32, D_MODEL // 2)),
        "macro_hidden_mult": 2,
        "macro_use_gated_update": True,
        "macro_update_scale": 0.05,
        "macro_update_scale_init": 0.1,
        "macro_include_delta_to_h0": True,
        "macro_use_depth_embedding": True,
        "macro_decomposition": "direct",
        "router_hidden": max(64, D_MODEL // 2),
    },
    "training": {
        "mode": "dense_exact",
        "optimizer": "adamw",
        "lr": 1e-3,
        "weight_decay": 0.1,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "grad_accum_steps": 1,
        "grad_clip_norm": 1.0,
        "seed": SEED,
        "precision": PRECISION,
        "audit_p_min": 0.0,
        "audit_p_max": 0.0,
        "audit_alpha": 0.0,
        "audit_beta": 0.0,
        "audit_gamma": 0.0,
        "audit_mode": "metric_only",
        "audit_gradient_correction": False,
        "lambda_cons": 0.001,
        "disable_router_aux": True,
        "debug_force_full_output": True,
        "coverage_min": 0.0,
        "log_every": 25,
        "compile_model": False,
        "fused_optimizer": False,
        "foreach_optimizer": False,
        "allow_tf32": True,
        "strict_cuda": True,
        "require_triton": False,
        "require_flash_attention": False,
    },
    "output": {"mode": "full"},
    "data": {
        "dataset": "tinystories" if USE_TINYSTORIES else "synthetic",
        "tokenizer": "gpt2_bpe",
        "vocab_projection": "filter",
        "projection_lane": "filter",
        "train_tokens": SYNTHETIC_TOKENS if USE_TINYSTORIES else None,
        "max_tokens": SYNTHETIC_TOKENS,
        "eval_tokens": EVAL_TOKENS,
        "synthetic_tokens": SYNTHETIC_TOKENS,
        "seed": SEED,
    },
}
# Drop None values so the dataclass defaults behave normally.
config["data"] = {k: v for k, v in config["data"].items() if v is not None}

CONFIG_PATH = CONFIG_DIR / f"static_cluster_ffn_d{D_MODEL}_h{D_FF}.yaml"
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print("CONFIG_PATH:", CONFIG_PATH)
print(CONFIG_PATH.read_text()[:2000])

## Train Or Load Dense Checkpoints

This is chunked so early handoff checkpoints at steps `100` and `200` exist. Each checkpoint includes optimizer state, which matters for clean continuation.

In [ ]:
dense_checkpoints: dict[int | str, Path] = {}

if DENSE_CHECKPOINT_OVERRIDE and not RUN_DENSE_TRAIN:
    DENSE_CHECKPOINT = Path(DENSE_CHECKPOINT_OVERRIDE).expanduser()
    if not DENSE_CHECKPOINT.exists():
        raise FileNotFoundError(f"DENSE_CHECKPOINT does not exist on this remote runtime: {DENSE_CHECKPOINT}")
    dense_checkpoints["override"] = DENSE_CHECKPOINT
    print("Using provided dense checkpoint:", DENSE_CHECKPOINT)
else:
    current_checkpoint: Path | None = None
    completed_steps = 0
    for target_step in sorted(set(DENSE_CKPT_STEPS)):
        if target_step <= completed_steps:
            continue
        ckpt = CHECKPOINT_DIR / f"dense_step{target_step}.pt"
        dense_checkpoints[target_step] = ckpt
        if RESUME_EXISTING and ckpt.exists():
            print(f"Reusing existing checkpoint for step {target_step}: {ckpt}")
            current_checkpoint = ckpt
            completed_steps = target_step
            continue
        chunk_steps = target_step - completed_steps
        args: list[object] = [
            "train",
            "--config", CONFIG_PATH,
            "--mode", "dense_exact",
            "--steps", chunk_steps,
            "--save-checkpoint", ckpt,
            "--run-dir", RUN_ROOT / f"dense_to_{target_step}",
        ]
        if current_checkpoint is not None:
            args.extend(["--resume", current_checkpoint])
        run_rte(*args)
        current_checkpoint = ckpt
        completed_steps = target_step

    DENSE_CHECKPOINT = dense_checkpoints[max(k for k in dense_checkpoints if isinstance(k, int))]
    if not DENSE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Dense checkpoint was not created: {DENSE_CHECKPOINT}")
    print("Final dense checkpoint:", DENSE_CHECKPOINT)

print("Dense checkpoints:")
for key, value in dense_checkpoints.items():
    print(" ", key, value)

## Static Cluster-Pool Continuation Helpers

The continuation command compares dense continuation against static sparse-FFN continuation with dense attention still enabled. It also logs gradient alignment, coverage, row-gradient norms, and sparse gaps versus dense at each eval step.

In [ ]:
def run_continuation(
    *,
    name: str,
    checkpoint: Path,
    steps: int,
    eval_steps: list[int],
    refresh_intervals: list[int],
    run_dense: bool = True,
    run_sparse: bool = True,
) -> Path:
    output = REPORT_DIR / f"{name}.json"
    if RESUME_EXISTING and output.exists():
        print("Reusing existing report:", output)
        return output
    args: list[object] = [
        "static-cluster-pool-continuation",
        "--config", CONFIG_PATH,
        "--dense-checkpoint", checkpoint,
        "--steps", steps,
        "--lr", CONT_LR,
        "--weight-decay", CONT_WEIGHT_DECAY,
        "--resume-optimizer-state",
        "--eval-batches", EVAL_BATCHES,
        "--eval-steps", *eval_steps,
        "--calibration-tokens", CALIBRATION_TOKENS,
        "--rank", SVD_RANK,
        "--clusters", CLUSTERS,
        "--candidate-m", CANDIDATE_M,
        "--refresh-intervals", *refresh_intervals,
        "--cluster-iters", CLUSTER_ITERS,
        "--include-gradient-alignment",
        "--alignment-batches", ALIGNMENT_BATCHES,
        "--alignment-batch-size", ALIGNMENT_BATCH_SIZE,
        "--output", output,
    ]
    args.append("--run-dense" if run_dense else "--no-run-dense")
    args.append("--run-sparse" if run_sparse else "--no-run-sparse")
    run_rte(*args)
    return output


continuation_reports: dict[str, Path] = {}

## 50-Step Sanity Gate

This is the quick check: dense must be stable, and static sparse should not widen the initial gap.

In [ ]:
if RUN_50_CONTINUATION:
    continuation_reports["final_50"] = run_continuation(
        name="final_checkpoint_50_steps",
        checkpoint=DENSE_CHECKPOINT,
        steps=50,
        eval_steps=[0, 50],
        refresh_intervals=[0, 50],
    )
else:
    print("Skipping 50-step continuation.")

## 300-Step Stability Gate

This is the main gate from the guide: dense continuation, no-refresh static sparse FFN, refresh@100, and refresh@50. Eval is logged at `0, 50, 100, 200, 300` over `65,536` tokens by default.

In [ ]:
if RUN_300_CONTINUATION:
    continuation_reports["final_300"] = run_continuation(
        name="final_checkpoint_300_steps",
        checkpoint=DENSE_CHECKPOINT,
        steps=300,
        eval_steps=[0, 50, 100, 200, 300],
        refresh_intervals=[0, 100, 50],
    )
else:
    print("Skipping 300-step continuation.")

## Forward + Backward CUDA Benchmark

This measures dense FFN versus static C16/M192 FFN forward+backward and includes the scatter cost. The target gate is at least `5x` for `d=2048,H=8192` including scatter, with `10x` as the strong gate.

In [ ]:
if RUN_FWD_BWD_BENCH:
    bench_output = REPORT_DIR / "static_cluster_pool_fwd_bwd_benchmark.json"
    if RESUME_EXISTING and bench_output.exists():
        print("Reusing existing benchmark:", bench_output)
    else:
        args: list[object] = [
            "benchmark-static-cluster-pool-ffn-train",
            "--clusters", CLUSTERS,
            "--candidate-m", CANDIDATE_M,
            "--iters", BENCH_ITERS,
            "--warmup", BENCH_WARMUP,
            "--output", bench_output,
        ]
        for size in BENCH_SIZES:
            args.extend(["--size", size])
        run_rte(*args)
else:
    bench_output = None
    print("Skipping forward+backward benchmark.")

## Early-Checkpoint Handoff

This tests whether sparse FFN works before the dense model is fully polished. It uses the chunked checkpoints created above, if available. If you supplied an external final checkpoint and did not train dense chunks in this notebook, this section skips cleanly.

In [ ]:
if RUN_EARLY_HANDOFF:
    for handoff_step in EARLY_HANDOFF_STEPS:
        ckpt = dense_checkpoints.get(handoff_step)
        if ckpt is None or not Path(ckpt).exists():
            print(f"Skipping early handoff at step {handoff_step}; checkpoint is unavailable.")
            continue
        continuation_reports[f"early_{handoff_step}_300"] = run_continuation(
            name=f"early_handoff_step{handoff_step}_{EARLY_HANDOFF_CONT_STEPS}_steps",
            checkpoint=Path(ckpt),
            steps=EARLY_HANDOFF_CONT_STEPS,
            eval_steps=[0, 50, 100, 200, EARLY_HANDOFF_CONT_STEPS],
            refresh_intervals=[0],
        )
else:
    print("Skipping early handoff tests.")

## Summary Tables

This cell reads the JSON reports and prints compact pass/fail-facing summaries. The full reports stay in the remote run directory.

In [ ]:
def _load_json(path: Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text())


def summarize_continuation(path: Path) -> None:
    report = _load_json(path)
    print("\n===", path.name, "===")
    rows = report.get("rows", [])
    dense = next((row for row in rows if row.get("variant") == "dense_continuation"), None)
    if dense:
        print(
            f"dense: {dense.get('initial_nll_per_token'):.6f} -> {dense.get('final_nll_per_token'):.6f} "
            f"delta={dense.get('nll_delta'):.6f}"
        )
    for row in rows:
        variant = row.get("variant")
        if variant == "dense_continuation":
            continue
        init = row.get("initial_nll_per_token")
        final = row.get("final_nll_per_token")
        init_gap = row.get("initial_gap_vs_dense")
        final_gap = row.get("final_gap_vs_dense")
        print(
            f"{variant}: {init:.6f} -> {final:.6f} "
            f"gap {init_gap:.6f} -> {final_gap:.6f}"
        )
        align_initial = row.get("initial_gradient_alignment") or {}
        align_final = row.get("final_gradient_alignment") or {}
        if align_initial:
            print(
                "  grad cos initial/final: input="
                f"{align_initial.get('input_grad_cosine')} / {align_final.get('input_grad_cosine')}"
            )
        coverage = (row.get("final_coverage") or {}).get("mean") or {}
        if coverage:
            print(
                "  coverage: unique="
                f"{coverage.get('unique_selected_neurons'):.2f}, "
                f"fraction={coverage.get('coverage_fraction'):.4f}"
            )
        curve = row.get("eval_curve") or []
        if curve:
            compact = [(int(x.get("step", 0)), round(float(x.get("nll_per_token", 0)), 5), x.get("gap_vs_dense")) for x in curve]
            print("  curve:", compact)


for name, path in continuation_reports.items():
    if Path(path).exists():
        summarize_continuation(Path(path))

if bench_output and Path(bench_output).exists():
    bench = _load_json(Path(bench_output))
    print("\n===", Path(bench_output).name, "===")
    for row in bench.get("rows", []):
        print(json.dumps({
            "size": f"{row['d_model']}x{row['d_ff']}x{row['tokens']}",
            "dense_fwd_bwd_ms": row.get("dense_forward_backward_ms"),
            "sparse_fwd_bwd_ms": row.get("sparse_forward_backward_ms"),
            "scatter_ms": row.get("grad_scatter_ms"),
            "speedup_no_scatter": row.get("forward_backward_speedup"),
            "speedup_with_scatter": row.get("forward_backward_scatter_speedup"),
            "peak_memory": row.get("peak_memory"),
        }, indent=2))

print("\nRun root:", RUN_ROOT)